# ◈ ARCANA — Stage 4A Rolling Window (v4)

### Key change from v3
- **3-year rolling training window** instead of expanding
- **Elite pairs re-selected each year** from that year's top pairs
- Pairs adapt to structural breaks — stale cointegrations get dropped automatically

```
Train: 2010–2012 → select pairs → trade 2013
Train: 2011–2013 → select pairs → trade 2014
Train: 2012–2014 → select pairs → trade 2015
...
Train: 2022–2024 → select pairs → trade 2025
```

### Outputs
- `signals_gated_rolling.csv` — regime-gated signals (elite pairs per year)
- `pairs_by_year_rolling.csv` — pair metadata per year with hedge ratios
- `hedge_ratios_rolling.csv` — hedge ratio per pair per year
---

## 0 · Imports & Config

In [ ]:
import os, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.stattools import coint
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant
from itertools import combinations
from collections import Counter
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'#0d0f14','axes.facecolor':'#0d0f14','axes.edgecolor':'#2a2d35',
    'axes.labelcolor':'#8b8fa8','axes.titlecolor':'#e2e4ed','axes.titlesize':13,
    'axes.labelsize':11,'axes.grid':True,'grid.color':'#1e2029','grid.linewidth':0.6,
    'xtick.color':'#555870','ytick.color':'#555870','xtick.labelsize':9,'ytick.labelsize':9,
    'text.color':'#e2e4ed','legend.facecolor':'#12141a','legend.edgecolor':'#2a2d35',
    'legend.fontsize':9,'lines.linewidth':1.4,'font.family':'monospace',
})
C_GOLD='#c9a84c';C_TEAL='#3ec9b0';C_BLUE='#4f8ef7'
C_RED='#e05c6b';C_PURPLE='#9b7fe8';C_GREY='#555870';C_WHITE='#e2e4ed'
REGIME_COLORS={0:C_TEAL,1:C_GOLD,2:C_RED}

STAGE1_DIR  = r'C:\Arbion Research\Stage 1 data layer\Universe_stock data'
STAGE3_DIR  = r'C:\Arbion Research\Stage 3 HMM regime engine'
STAGE4A_DIR = r'C:\Arbion Research\Stage 4A stat arb engine'
OUTPUT_DIR  = r'C:\Arbion Research\Stage 4A stat arb engine'

START_DATE       = '2010-01-01'
END_DATE         = '2025-12-31'
FIRST_TRADE_YEAR = 2013
ROLLING_YEARS    = 3      # training window length

COINT_PVAL    = 0.05
MIN_HALFLIFE  = 5
MAX_HALFLIFE  = 80      # wider — allows more pairs in later years
MIN_OU_R2     = 0.05
ZSCORE_WINDOW = 60
ZSCORE_ENTRY  = 2.0
ZSCORE_EXIT   = 0.5
TOP_PAIRS_PER_YEAR = 10  # elite pairs selected fresh each year
REGIME_SCALE  = {0:1.0, 1:0.7, 2:0.0}

SECTOR_MAP = {
    'AAPL':'Tech','MSFT':'Tech','NVDA':'Tech','AVGO':'Tech','ORCL':'Tech',
    'CRM':'Tech','ADBE':'Tech','AMD':'Tech','INTC':'Tech','QCOM':'Tech',
    'TXN':'Tech','IBM':'Tech','ACN':'Tech','AMAT':'Tech','CTSH':'Tech','MU':'Tech',
    'BRK-B':'Financials','JPM':'Financials','BAC':'Financials','WFC':'Financials',
    'GS':'Financials','MS':'Financials','BLK':'Financials','SCHW':'Financials',
    'AXP':'Financials','C':'Financials','USB':'Financials','PNC':'Financials',
    'COF':'Financials','CB':'Financials','ICE':'Financials',
    'LLY':'Healthcare','UNH':'Healthcare','JNJ':'Healthcare','ABBV':'Healthcare',
    'MRK':'Healthcare','TMO':'Healthcare','ABT':'Healthcare','DHR':'Healthcare',
    'BMY':'Healthcare','AMGN':'Healthcare','GILD':'Healthcare','CVS':'Healthcare',
    'CI':'Healthcare','HCA':'Healthcare','SYK':'Healthcare',
    'AMZN':'ConsDisc','TSLA':'ConsDisc','HD':'ConsDisc','MCD':'ConsDisc',
    'NKE':'ConsDisc','LOW':'ConsDisc','SBUX':'ConsDisc','TJX':'ConsDisc',
    'BKNG':'ConsDisc','CMG':'ConsDisc','MAR':'ConsDisc','F':'ConsDisc',
    'GM':'ConsDisc','ORLY':'ConsDisc','AZO':'ConsDisc',
    'GOOG':'Comms','META':'Comms','NFLX':'Comms','DIS':'Comms',
    'CMCSA':'Comms','T':'Comms','VZ':'Comms','CHTR':'Comms','EBAY':'Comms','EA':'Comms',
    'GE':'Industrials','CAT':'Industrials','HON':'Industrials','UNP':'Industrials',
    'RTX':'Industrials','BA':'Industrials','LMT':'Industrials','DE':'Industrials',
    'ETN':'Industrials','MMM':'Industrials','FDX':'Industrials','UPS':'Industrials',
    'WM':'Industrials','NSC':'Industrials','EMR':'Industrials',
    'WMT':'Staples','PG':'Staples','KO':'Staples','PEP':'Staples','COST':'Staples',
    'PM':'Staples','MO':'Staples','CL':'Staples','MDLZ':'Staples','KHC':'Staples',
    'XOM':'Energy','CVX':'Energy','COP':'Energy','EOG':'Energy','SLB':'Energy',
    'MPC':'Energy','PSX':'Energy','VLO':'Energy','OXY':'Energy','KMI':'Energy',
    'AMT':'REIT','SPY':'ETF'
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('✓ Config loaded')
print(f'  Rolling window   : {ROLLING_YEARS} years')
print(f'  Top pairs/year   : {TOP_PAIRS_PER_YEAR} (re-selected each year)')
print(f'  Half-life range  : {MIN_HALFLIFE}–{MAX_HALFLIFE}d')
print(f'  Z-score window   : {ZSCORE_WINDOW}d rolling')

---
## Step 1 · Load Log Prices + Build Sector Pairs

In [ ]:
print('Loading prices...')
price_dict={}
for fpath in glob.glob(os.path.join(STAGE1_DIR,'*.csv')):
    ticker=os.path.basename(fpath).replace('.csv','')
    try:
        df=pd.read_csv(fpath,parse_dates=['Date'])
        df=df.sort_values('Date').set_index('Date')
        close_col=next((c for c in ['Adj Close','Adj. Close','Close','close']
                        if c in df.columns),None)
        if close_col: price_dict[ticker]=df[close_col].astype(float)
    except: pass

prices_raw=pd.DataFrame(price_dict)
prices_raw.index=pd.to_datetime(prices_raw.index)
prices_raw=prices_raw.sort_index().loc[START_DATE:END_DATE]
prices_raw=prices_raw.replace(0,np.nan).dropna(how='all')
log_prices=np.log(prices_raw)
common_dates=log_prices.diff().dropna(how='all').index

regime_labels=pd.read_csv(os.path.join(STAGE3_DIR,'regime_labels_weekly.csv'),
                           index_col=0,parse_dates=True).squeeze()
regime_labels=regime_labels.reindex(common_dates,method='ffill')

# Sector pairs
sector_pairs=[]
for sector in sorted(set(v for v in SECTOR_MAP.values() if v!='ETF')):
    members=sorted([t for t,s in SECTOR_MAP.items()
                    if s==sector and t in log_prices.columns])
    if len(members)<2: continue
    for tA,tB in combinations(members,2):
        sector_pairs.append((tA,tB,sector))

print(f'✓ {len(log_prices.columns)} tickers | {len(sector_pairs)} sector pairs')
print(f'  Date range: {common_dates[0].date()} to {common_dates[-1].date()}')
print(f'\nSector breakdown:')
for sector,n in sorted(Counter(s for _,_,s in sector_pairs).items()):
    members=[t for t,s in SECTOR_MAP.items() if s==sector and t in log_prices.columns]
    print(f'  {sector:<20} {len(members):>3} stocks  {n:>4} pairs')

---
## Step 2 · OU Helper

In [ ]:
def fit_ou_mle(spread):
    s=spread.dropna().values.astype(float)
    if len(s)<30:
        return {'theta':np.nan,'mu':np.nan,'sigma':np.nan,'half_life':np.nan,'r2':np.nan}
    s_t=s[1:];s_l=s[:-1]
    X=np.column_stack([np.ones(len(s_l)),s_l])
    a,b=np.linalg.lstsq(X,s_t,rcond=None)[0]
    if b<=0 or b>=1:
        return {'theta':np.nan,'mu':np.nan,'sigma':np.nan,'half_life':np.nan,'r2':np.nan}
    theta=-np.log(b);mu=a/(1-b)
    resid_ar=s_t-(a+b*s_l);sigma=float(np.std(resid_ar))
    half_life=np.log(2)/theta
    ss_res=float(np.sum(resid_ar**2));ss_tot=float(np.sum((s_t-s_t.mean())**2))
    r2=1-ss_res/ss_tot if ss_tot>0 else 0
    return {'theta':theta,'mu':mu,'sigma':sigma,'half_life':half_life,'r2':r2}

print('✓ OU helper defined')

---
## Step 3 · Rolling Walk-Forward Loop
⚠ Expected runtime: 20–40 minutes

In [ ]:
trade_years  = list(range(FIRST_TRADE_YEAR,2026))
zscore_wf    = pd.DataFrame(np.nan,index=common_dates,columns=[])
spreads_wf   = pd.DataFrame(np.nan,index=common_dates,columns=[])
pairs_log    = []   # all pairs across all years
elite_by_year= {}  # which pairs were elite in each trade year

print(f'Rolling walk-forward: {len(trade_years)} trade years')
print(f'Training window: {ROLLING_YEARS} years rolling')
print(f'Elite pairs re-selected fresh each year\n')

for trade_year in trade_years:
    # Rolling window — fixed 3-year lookback
    train_start = f'{trade_year - ROLLING_YEARS}-01-01'
    train_end   = f'{trade_year-1}-12-31'
    trade_start = f'{trade_year}-01-01'
    trade_end   = f'{trade_year}-12-31'

    train_idx = common_dates[(common_dates >= train_start) & (common_dates <= train_end)]
    trade_idx = common_dates[(common_dates >= trade_start) & (common_dates <= trade_end)]
    if len(train_idx) < 252 or len(trade_idx) == 0:
        print(f'  {trade_year}: skipped (insufficient data)'); continue

    lp_train = log_prices.loc[train_start:train_end]

    # ── Cointegration on log prices ───────────────────────────────────────
    coint_pass=[]
    for tA,tB,sector in sector_pairs:
        if tA not in lp_train.columns or tB not in lp_train.columns: continue
        s1=lp_train[tA].dropna()
        s2=lp_train[tB].dropna()
        idx=s1.index.intersection(s2.index)
        if len(idx)<252: continue
        try:
            _,pval,_=coint(s1.loc[idx],s2.loc[idx])
            if pval>=COINT_PVAL: continue
            X=add_constant(s2.loc[idx].values)
            h=OLS(s1.loc[idx].values,X).fit().params[1]
            coint_pass.append((tA,tB,sector,pval,h))
        except: pass

    # ── OU fitting ────────────────────────────────────────────────────────
    selected=[]
    for tA,tB,sector,pval,h in coint_pass:
        s1=lp_train[tA].dropna()
        s2=lp_train[tB].dropna()
        idx=s1.index.intersection(s2.index)
        spread_train=pd.Series(
            s1.loc[idx].values-h*s2.loc[idx].values,index=idx)
        ou=fit_ou_mle(spread_train)
        if (ou['half_life'] and MIN_HALFLIFE<=ou['half_life']<=MAX_HALFLIFE
                and ou['r2'] and ou['r2']>=MIN_OU_R2):
            selected.append({'tA':tA,'tB':tB,'sector':sector,'h':h,
                             'pval':pval,**ou})

    # ── Select elite pairs for THIS year ─────────────────────────────────
    # Quality = R² / half_life — pick top N for this trade year
    if selected:
        sel_df=pd.DataFrame(selected)
        sel_df['quality']=sel_df['r2']/sel_df['half_life']
        sel_df=sel_df.sort_values('quality',ascending=False)
        year_elite=sel_df.head(TOP_PAIRS_PER_YEAR)
    else:
        year_elite=pd.DataFrame()

    elite_by_year[trade_year]=year_elite['tA'].str.cat(
        year_elite['tB'],sep='_').tolist() if len(year_elite)>0 else []

    # ── Rolling Z-score for elite pairs in trade year ─────────────────────
    lp_trade=log_prices.loc[:trade_end]
    n_signals=0

    for _,p in year_elite.iterrows():
        tA,tB,h=p['tA'],p['tB'],p['h']
        pair=f'{tA}_{tB}'
        if tA not in lp_trade.columns or tB not in lp_trade.columns: continue

        s1=lp_trade[tA].dropna()
        s2=lp_trade[tB].dropna()
        idx_full=s1.index.intersection(s2.index)
        spread_full=pd.Series(
            s1.loc[idx_full].values-h*s2.loc[idx_full].values,index=idx_full)

        roll_mean=spread_full.rolling(ZSCORE_WINDOW).mean()
        roll_std =spread_full.rolling(ZSCORE_WINDOW).std()
        zscore_full=(spread_full-roll_mean)/roll_std.replace(0,np.nan)

        trade_mask=(idx_full>=trade_start)&(idx_full<=trade_end)
        idx_trade=idx_full[trade_mask]
        if len(idx_trade)==0: continue

        if pair not in zscore_wf.columns:
            zscore_wf[pair] =np.nan
            spreads_wf[pair]=np.nan
        zscore_wf.loc[idx_trade,pair] =zscore_full.loc[idx_trade].values
        spreads_wf.loc[idx_trade,pair]=spread_full.loc[idx_trade].values
        n_signals+=1

        pairs_log.append({
            'trade_year':trade_year,'pair':pair,'sector':p['sector'],
            'tA':tA,'tB':tB,'h':h,'pval':p['pval'],
            'half_life':p['half_life'],'r2':p['r2'],'quality':p['quality'],
            'is_elite':True
        })

    pairs_str=', '.join(elite_by_year[trade_year][:5])
    print(f'  {trade_year}: window={train_start[:4]}-{train_end[:4]}  '
          f'coint={len(coint_pass):>3}  elite={len(year_elite):>2}  '
          f'signals={n_signals:>2}  | {pairs_str}{"..." if len(elite_by_year[trade_year])>5 else ""}')

# Save
zscore_wf.to_csv(os.path.join(OUTPUT_DIR,'zscore_signals_rolling.csv'))
spreads_wf.to_csv(os.path.join(OUTPUT_DIR,'spreads_rolling.csv'))
pd.DataFrame(pairs_log).to_csv(os.path.join(OUTPUT_DIR,'pairs_by_year_rolling.csv'),index=False)

print(f'\n✓ Rolling walk-forward complete')
print(f'  Total pair-years : {len(pairs_log)}')
if pairs_log:
    plog=pd.DataFrame(pairs_log)
    print(f'  Unique pairs     : {plog["pair"].nunique()}')
print(f'  Z-score shape    : {zscore_wf.shape}')

---
## Step 4 · Regime Gating + Save
Safe to run after kernel restart — reloads from disk.

In [ ]:
import os,pandas as pd,numpy as np
C_GOLD='#c9a84c';C_TEAL='#3ec9b0';C_BLUE='#4f8ef7'
C_RED='#e05c6b';C_PURPLE='#9b7fe8';C_GREY='#555870';C_WHITE='#e2e4ed'
OUTPUT_DIR=r'C:\Arbion Research\Stage 4A stat arb engine'
STAGE3_DIR=r'C:\Arbion Research\Stage 3 HMM regime engine'
REGIME_SCALE={0:1.0,1:0.7,2:0.0}

plog=pd.read_csv(os.path.join(OUTPUT_DIR,'pairs_by_year_rolling.csv'))
zscore_wf=pd.read_csv(os.path.join(OUTPUT_DIR,'zscore_signals_rolling.csv'),
                       index_col=0,parse_dates=True)
regime_labels=pd.read_csv(os.path.join(STAGE3_DIR,'regime_labels_weekly.csv'),
                           index_col=0,parse_dates=True).squeeze()
regime_labels=regime_labels.reindex(zscore_wf.index,method='ffill')

# Regime gating — applied to all pairs
signals_gated=zscore_wf.copy()
rl=regime_labels.values
for i,regime in enumerate(rl):
    if pd.isna(regime): continue
    signals_gated.iloc[i]=signals_gated.iloc[i]*REGIME_SCALE[int(regime)]

signals_gated.to_csv(os.path.join(OUTPUT_DIR,'signals_gated_rolling.csv'))

# Save hedge ratios per pair per year (for Stage 6)
plog[['trade_year','pair','tA','tB','h','half_life','r2']].to_csv(
    os.path.join(OUTPUT_DIR,'hedge_ratios_rolling.csv'),index=False)

print(f'✓ signals_gated_rolling.csv — shape {signals_gated.shape}')
print(f'✓ hedge_ratios_rolling.csv')
print(f'\nPairs active per year:')
print(f'{"Year":<6} {"Pairs":>6} {"Unique pairs"}')
for year in sorted(plog["trade_year"].unique()):
    yr=plog[plog["trade_year"]==year]
    print(f'  {year:<6} {len(yr):>4}   {" ".join(yr["pair"].tolist())}')

---
## Step 5 · Visualise Elite Pairs by Year

In [ ]:
import matplotlib.pyplot as plt,matplotlib.dates as mdates
C_GOLD='#c9a84c';C_TEAL='#3ec9b0';C_BLUE='#4f8ef7'
C_RED='#e05c6b';C_PURPLE='#9b7fe8';C_GREY='#555870';C_WHITE='#e2e4ed'
OUTPUT_DIR=r'C:\Arbion Research\Stage 4A stat arb engine'

pairs_per_year=plog.groupby('trade_year').size()
all_pairs=sorted(plog['pair'].unique())

# Heatmap: which pairs were active in which years
presence=pd.DataFrame(0,index=all_pairs,
                       columns=sorted(plog['trade_year'].unique()))
for _,row in plog.iterrows():
    presence.loc[row['pair'],row['trade_year']]=1

fig,axes=plt.subplots(1,2,figsize=(16,max(6,len(all_pairs)*0.3+2)))
fig.suptitle('ARCANA  ·  Rolling Pair Selection — Active Pairs by Year',
             color=C_GOLD,fontsize=14)

# Pair presence heatmap
ax=axes[0]
im=ax.imshow(presence.values,aspect='auto',cmap='YlOrRd',vmin=0,vmax=1)
ax.set_yticks(range(len(all_pairs)))
ax.set_yticklabels(all_pairs,fontsize=7,color=C_WHITE)
ax.set_xticks(range(len(presence.columns)))
ax.set_xticklabels(presence.columns,fontsize=8,rotation=45,color=C_WHITE)
ax.set_title('Pair active = orange  |  inactive = black',color=C_WHITE,fontsize=9)
ax.set_xlabel('Trade year')

# Bar: pairs per year
ax2=axes[1]
ax2.bar(pairs_per_year.index,pairs_per_year.values,color=C_TEAL,alpha=0.8)
for y,n in pairs_per_year.items():
    ax2.text(y,n+0.1,str(n),ha='center',fontsize=9,color=C_GOLD)
ax2.set_xlabel('Trade year');ax2.set_ylabel('Elite pairs active')
ax2.set_title(f'Elite pairs per year (top {TOP_PAIRS_PER_YEAR} re-selected annually)',
              color=C_WHITE,fontsize=9)
ax2.set_ylim(0,TOP_PAIRS_PER_YEAR+2)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'viz_rolling_pairs_by_year.png'),
            dpi=150,bbox_inches='tight',facecolor='#0d0f14')
plt.show();print('✓ viz_rolling_pairs_by_year.png')

In [ ]:
# Z-score for top 4 most-frequent pairs
top4=plog.groupby('pair').size().sort_values(ascending=False).head(4).index.tolist()
n_show=len(top4)
fig,axes=plt.subplots(n_show,1,figsize=(15,3*n_show),sharex=True)
if n_show==1: axes=[axes]
fig.suptitle('ARCANA  ·  Rolling Z-Score — Most Frequent Elite Pairs',
             color=C_GOLD,fontsize=14)

for i,pair in enumerate(top4):
    ax=axes[i]
    z=zscore_wf[pair].dropna() if pair in zscore_wf.columns else pd.Series(dtype=float)
    if len(z)==0: continue
    colors_z=np.where(z.values>2,C_RED,np.where(z.values<-2,C_TEAL,C_GREY))
    ax.plot(z.index,z.values,color=C_WHITE,linewidth=0.5,alpha=0.4)
    ax.scatter(z.index,z.values,c=colors_z,s=2,alpha=0.6,rasterized=True)
    ax.axhline(2,color=C_RED,linewidth=1.0,linestyle='--',alpha=0.7)
    ax.axhline(-2,color=C_TEAL,linewidth=1.0,linestyle='--',alpha=0.7)
    ax.axhline(0,color=C_GREY,linewidth=0.4)
    yrs=sorted(plog[plog['pair']==pair]['trade_year'].tolist())
    ax.set_title(f'{pair}  — active years: {yrs}',color=C_WHITE,fontsize=9)
    ax.set_ylabel('Z-score');ax.set_ylim(-5,5)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'viz_rolling_zscore.png'),
            dpi=150,bbox_inches='tight',facecolor='#0d0f14')
plt.show();print('✓ viz_rolling_zscore.png')

---
## Step 6 · Summary

In [ ]:
print('='*60)
print('  ARCANA — Stage 4A Rolling Window Complete')
print('='*60)
print(f'  Rolling window   : {ROLLING_YEARS} years')
print(f'  Top pairs/year   : {TOP_PAIRS_PER_YEAR} (re-selected each year)')
print(f'  Total pair-years : {len(plog)}')
print(f'  Unique pairs     : {plog["pair"].nunique()}')
print(f'  Signal shape     : {signals_gated.shape}')
print(f'\n  Pairs by year:')
for year in sorted(plog["trade_year"].unique()):
    yr=plog[plog["trade_year"]==year]
    print(f'    {year}: {list(yr["pair"])}')
print(f'\n  Output files:')
for fname in ['zscore_signals_rolling.csv','signals_gated_rolling.csv',
              'hedge_ratios_rolling.csv','pairs_by_year_rolling.csv']:
    fpath=os.path.join(OUTPUT_DIR,fname)
    size=os.path.getsize(fpath)/1024 if os.path.exists(fpath) else 0
    print(f'    {fname:<35} {size:>8.0f} KB')
print(f'\n  Next: Stage 5 simplified → Stage 6 v2')
print('='*60)